# Six Sigma DMAIC — Analyze Phase

Pareto analysis to prioritize the improvement target, plus stratification
by category and priority.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

%matplotlib inline
sns.set_style("whitegrid")

PROJECT_ROOT = Path.cwd().parent
RAW = PROJECT_ROOT / "data" / "raw"
CHARTS = PROJECT_ROOT / "docs" / "screenshots"

df = pd.read_csv(RAW / "helpdesk_tickets.csv", parse_dates=["created_at"])

### Pareto by total impact (sum of resolution hours, not just ticket count)

In [ ]:
# Prioritizing by total hours consumed, not ticket count, is the correct
# Six Sigma approach -- a category can have few tickets but still dominate
# total process time.
pareto = df.groupby("category")["resolution_hours"].sum().sort_values(ascending=False)
pareto_pct = pareto / pareto.sum() * 100
pareto_cum = pareto_pct.cumsum()

for cat in pareto.index:
    print(f"{cat:22s} {pareto[cat]:>8,.0f}h  ({pareto_pct[cat]:5.1f}%)  cum: {pareto_cum[cat]:5.1f}%")

In [ ]:
fig, ax1 = plt.subplots(figsize=(10, 5))
ax1.bar(pareto.index, pareto.values, color="#4C72B0", edgecolor="white")
ax1.set_ylabel("Total hours consumed")
ax1.tick_params(axis="x", rotation=30)

ax2 = ax1.twinx()
ax2.plot(pareto.index, pareto_cum.values, color="#C44E52", marker="o", linewidth=2)
ax2.axhline(80, color="gray", linestyle="--", linewidth=1)
ax2.set_ylabel("Cumulative %")
ax2.set_ylim(0, 105)

ax1.set_title("Pareto -- Process Time by Category", fontsize=12, pad=10)
plt.tight_layout()
plt.savefig(CHARTS / "pareto_chart.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"Network alone accounts for {pareto_pct.iloc[0]:.1f}% of all process time.")

### Stratification -- is Network's delay driven by priority level?

In [ ]:
network = df[df["category"] == "Network"]

fig, ax = plt.subplots(figsize=(8, 4.5))
order = ["Low", "Medium", "High", "Critical"]
sns.boxplot(data=network, x="priority", y="resolution_hours", order=order, ax=ax, color="#4C72B0")
ax.set_title("Network Resolution Time by Priority", fontsize=12, pad=10)
ax.set_xlabel("Priority")
ax.set_ylabel("Resolution time (hours)")
plt.tight_layout()
plt.savefig(CHARTS / "stratification_by_priority.png", dpi=150, bbox_inches="tight")
plt.show()

print(network.groupby("priority")["resolution_hours"].agg(["count", "mean"]).round(1).reindex(order))

### Root cause categorization

Documented qualitatively in `dmaic/03_analyze.md` (5-Why / fishbone
style). Top contributing factors identified for Network tickets:
escalation delays to third-party ISPs, insufficient first-line
diagnostics documentation, and lack of a standard triage checklist.